# Spark: сравнение ресурсов

Идея: запускаем одинаковые действия на одном датасете, но меняем ресурсы Spark-приложения. Сравнивайте время выполнения, Spark UI `Jobs/Stages/Executors`, spill и количество task'ов.

## Как переключать размер кластера

Перед запуском notebook поднимите один из профилей:

```bash
docker compose --env-file profiles/spark-small.env up -d --scale spark-worker=1
docker compose --env-file profiles/spark-medium.env up -d --scale spark-worker=2
docker compose --env-file profiles/spark-large.env up -d --scale spark-worker=4
```

Spark Master UI: http://localhost:8080. Driver UI обычно: http://localhost:4040.

In [1]:
import sys
sys.path.append("/opt/workspace/scripts")

from spark_lab import make_spark, benchmark_groupby, benchmark_join, benchmark_skew

## Профиль приложения

Меняйте `PROFILE`. Важно: если SparkSession уже создана, сначала выполните `spark.stop()`, потом создайте новую сессию.

In [2]:
PROFILES = {
    "tiny": {
        "executor_memory": "512m",
        "executor_cores": 1,
        "cores_max": 1,
        "shuffle_partitions": 8,
    },
    "normal": {
        "executor_memory": "1g",
        "executor_cores": 1,
        "cores_max": 2,
        "shuffle_partitions": 16,
    },
    "wide": {
        "executor_memory": "2g",
        "executor_cores": 2,
        "cores_max": 4,
        "shuffle_partitions": 32,
    },
}

PROFILE = "normal"
conf = PROFILES[PROFILE]
conf

{'executor_memory': '1g',
 'executor_cores': 1,
 'cores_max': 2,
 'shuffle_partitions': 16}

In [3]:
spark = make_spark(app_name=f"resource_lab_{PROFILE}", **conf)
print("Spark UI:", "http://localhost:4040")
print("defaultParallelism:", spark.sparkContext.defaultParallelism)
print("shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/04 22:11:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark UI: http://localhost:4040
defaultParallelism: 2
shuffle.partitions: 16


In [4]:
BASE = "data/lab"
EVENTS = f"{BASE}/events"
EVENTS_PARTITIONED = f"{BASE}/events_partitioned"
USERS = f"{BASE}/users"
SKEWED = f"{BASE}/skewed_events"

## CPU/shuffle: group by

Здесь обычно видна разница между `cores_max=1/2/4` и количеством shuffle partitions.

In [5]:
benchmark_groupby(spark, EVENTS)

[Stage 2:================================>                         (9 + 2) / 16]

60
group by country/date: 5.83 sec


## Join: sort-merge против broadcast

Первый запуск запрещает broadcast, второй явно broadcast'ит маленький справочник.

In [6]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
benchmark_join(spark, EVENTS, USERS, broadcast=False)

+-------+--------------------+
|segment|              amount|
+-------+--------------------+
|    vip|1.6664999999999987E7|
|    new|1.6666666099999992E7|
| active|1.6663333899999997E7|
+-------+--------------------+

join broadcast=False: 4.95 sec


In [7]:
benchmark_join(spark, EVENTS, USERS, broadcast=True)

+-------+--------------------+
|segment|              amount|
+-------+--------------------+
|    vip|1.6665000000000004E7|
|    new|1.6666666100000106E7|
| active|1.6663333899999917E7|
+-------+--------------------+

join broadcast=True: 2.70 sec


## Partition pruning

Сравните чтение обычного и партиционированного датасета. В Spark UI в SQL-плане должно быть видно, что читается меньше файлов.

In [8]:
from spark_lab import timed

with timed("plain read with date filter"):
    print(spark.read.parquet(EVENTS).where("event_date = '2026-01-10'").count())

with timed("partitioned read with date filter"):
    print(spark.read.parquet(EVENTS_PARTITIONED).where("event_date = '2026-01-10'").count())

16667
plain read with date filter: 1.27 sec


16667
partitioned read with date filter: 1.32 sec


## Skew

На skewed key часть task'ов будет заметно дольше. Это хороший сценарий для объяснения, почему “добавить ресурсов” не всегда лечит проблему.

In [9]:
benchmark_skew(spark, SKEWED)

[Stage 35:=============================>                            (1 + 1) / 2]

skewed group by: 1.17 sec


In [ ]:
spark.stop()